In [0]:
import zipfile

with zipfile.ZipFile("/Volumes/workspace/default/raw/Taco Bueno - Pricing.zip") as z:
    archivos = [n for n in z.namelist() if n.endswith(".csv") and "__MACOSX" not in n]
    z.extractall("/Volumes/workspace/default/raw/csv/", members=archivos)
    print(len(archivos))

In [0]:
import re

df = (spark.read
      .option("header", True)
      .option("enforceSchema", False)
      .option("recursiveFileLookup", True)
      .csv("/Volumes/workspace/default/raw/csv/")
      .select("*", "_metadata.file_name"))

df = df.toDF(*[re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower() for c in df.columns])

df.write.mode("overwrite").saveAsTable("workspace.default.bronze_ventas")

In [0]:
import glob

encabezados = {}
for r in glob.glob("/Volumes/workspace/default/raw/csv/**/*.csv", recursive=True):
    with open(r, encoding="utf-8", errors="replace") as f:
        encabezados.setdefault(f.readline().rstrip("\r\n"), []).append(r.split("/")[-1])

for h, archivos in encabezados.items():
    archivos = sorted(archivos, key=lambda n: re.search(r"(\d{2})-(\d{2})-(\d{4})", n).group(3, 1, 2))
    print(len(archivos), "archivos:", repr(h))
    print(archivos[0], archivos[-1], "\n")